In [2]:
# ── 셀 1 │ 설정 (교체본 2: 그래프 조립 직접 구현) ──
import os, sys, json, random, datetime
import numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.utils import add_self_loops, softmax
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
from sklearn.model_selection import train_test_split
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

NEW       = 'C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/'
FEAT_DIR  = NEW + '01_pre-train/'
HERG_POOL = NEW + 'data/processed/herg_karim/herg_karim_pool.csv'
CLIN_POOL = NEW + 'data/processed/clinical/clinical_pool.csv'
GOUT      = NEW + 'data/graphs/'
CKPT      = NEW + 'models/pretrain/'
os.makedirs(GOUT, exist_ok=True); os.makedirs(CKPT, exist_ok=True)

if FEAT_DIR not in sys.path: sys.path.insert(0, FEAT_DIR)
import featurizer as FZ

ATOM_DIM, BOND_DIM = FZ.num_atom_features(), FZ.num_bond_features()
assert (ATOM_DIM, BOND_DIM) == (39, 10), f'차원이 39/10 이 아님: {ATOM_DIM}/{BOND_DIM}'
print('ATOM_DIM', ATOM_DIM, 'BOND_DIM', BOND_DIM)

def _prep(smi):
    mol = Chem.MolFromSmiles(str(smi))
    if mol is None or mol.GetNumAtoms() == 0 or mol.GetNumBonds() == 0:
        return None
    try: Chem.AssignStereochemistry(mol, cleanIt=True, force=True)
    except Exception: pass
    return mol

def make_graph(smi, y=None):
    """39차원 노드 / 10차원 엣지 그래프. 실패 시 None.
    무방향 그래프이므로 결합 하나를 양방향 두 엣지로 넣는다."""
    mol = _prep(smi)
    if mol is None:
        return None
    try:
        x = np.stack([FZ.atom_features(a) for a in mol.GetAtoms()])
    except Exception:
        return None          # degree>5 등 one_of_k_encoding 예외
    src, dst, eattr = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bf = FZ.bond_features(b)
        src += [i, j]; dst += [j, i]; eattr += [bf, bf]
    if not src:
        return None
    g = Data(x=torch.tensor(np.asarray(x, dtype=np.float32)),
             edge_index=torch.tensor([src, dst], dtype=torch.long),
             edge_attr=torch.tensor(np.asarray(eattr, dtype=np.float32)))
    if y is not None:
        g.y = torch.tensor([float(y)], dtype=torch.float)
    return g

def inchikey14(smi):
    mol = _prep(smi)
    if mol is None: return None
    try: return Chem.MolToInchiKey(mol).split('-')[0]
    except Exception: return None

SEED = 42
HID, HEADS, DROP = 32, 8, 0.3
EPOCHS, PATIENCE = 60, 12
LR_BB, LR_CLF, WD = 1e-4, 5e-3, 5e-4
BS_TR, BS_EV = 32, 64
DEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all()
print('device', DEV)

for smi in ['CN1CCC[C@H]1c1cccnc1', 'CC(=O)Oc1ccccc1C(=O)O', 'C', 'not_a_smiles']:
    g = make_graph(smi, 1.0)
    print(f'  {smi:<30} -> ' + ('None' if g is None
          else f'x={tuple(g.x.shape)} ei={tuple(g.edge_index.shape)} '
               f'ea={tuple(g.edge_attr.shape)} key={inchikey14(smi)}'))

ATOM_DIM 39 BOND_DIM 10
device cuda
  CN1CCC[C@H]1c1cccnc1           -> x=(12, 39) ei=(2, 26) ea=(26, 10) key=SNICXCGAKADSCV
  CC(=O)Oc1ccccc1C(=O)O          -> x=(13, 39) ei=(2, 26) ea=(26, 10) key=BSYNRYMUTXBXSQ
  C                              -> None
  not_a_smiles                   -> None


In [5]:
# ── 셀 2 │ 그래프 생성 (캐시) ──
GPATH = GOUT + 'herg_karim_graphs.pt'

if os.path.exists(GPATH):
    blob = torch.load(GPATH, weights_only=False)
    graphs, keys = blob['graphs'], blob['keys']
    print(f'캐시 로드: {len(graphs)}개')
else:
    pool = pd.read_csv(HERG_POOL)
    print('풀', len(pool), '컬럼', list(pool.columns))

    SMI = next(c for c in ['canon_smiles', 'smiles', 'Drug', 'SMILES']
               if c in pool.columns)
    YCOL = next(c for c in ['y', 'Y', 'label', 'Label', 'activity']
                if c in pool.columns)
    KCOL = next(c for c in ['key14', 'inchikey14', 'InChIKey14']
                if c in pool.columns)
    print(f'→ SMILES={SMI}  label={YCOL}  key={KCOL}')

    graphs, keys, fail = [], [], 0
    for i, (smi, k, y) in enumerate(zip(pool[SMI], pool[KCOL], pool[YCOL])):
        g = make_graph(smi, float(y))
        if g is None:
            fail += 1; continue
        graphs.append(g); keys.append(k)
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(pool)}  실패 {fail}')
    torch.save({'graphs': graphs, 'keys': keys}, GPATH)
    print(f'\n생성 {len(graphs)} / 실패 {fail}  → {GPATH}')

ys = np.array([int(g.y.item()) for g in graphs])
print('라벨 분포', dict(zip(*np.unique(ys, return_counts=True))), f'  양성률 {ys.mean():.3f}')
print('노드 특징 차원', graphs[0].x.shape[1], '/ 결합', graphs[0].edge_attr.shape[1])

풀 11037 컬럼 ['Drug_ID', 'Drug', 'canon_smiles', 'key14', 'Y', 'n_atoms']
→ SMILES=canon_smiles  label=Y  key=key14
  2000/11037  실패 0
  4000/11037  실패 0
  6000/11037  실패 0
  8000/11037  실패 0
  10000/11037  실패 0

생성 11037 / 실패 0  → C:/Users/iisl/Desktop/0_Molecualar_Informatics/2026_final/2026_new/data/graphs/herg_karim_graphs.pt
라벨 분포 {0: 5610, 1: 5427}   양성률 0.492
노드 특징 차원 39 / 결합 10


In [6]:
# ── 셀 3 │ 누수 제거 + split ──
# 임상 풀과 겹치는 분자는 사전학습 풀에서만 뺀다 (target test는 건드리지 않음).
clin_keys = set(pd.read_csv(CLIN_POOL)['key14'].dropna())
mask = np.array([k not in clin_keys for k in keys])
print(f'임상 중첩 {int((~mask).sum())}개 제거 → {int(mask.sum())}개')

gs = [g for g, m in zip(graphs, mask) if m]
ks = [k for k, m in zip(keys, mask) if m]
yy = ys[mask]

idx = np.arange(len(gs))
tr_i, tmp_i = train_test_split(idx, test_size=0.2, stratify=yy, random_state=SEED)
va_i, te_i  = train_test_split(tmp_i, test_size=0.5, stratify=yy[tmp_i], random_state=SEED)

# key14 단위 비중첩 검증 (TDC 기본 split의 분자 단위 오염을 되풀이하지 않기 위해)
kt, kv, ke = {ks[i] for i in tr_i}, {ks[i] for i in va_i}, {ks[i] for i in te_i}
assert not (kt & kv) and not (kt & ke) and not (kv & ke), '분자 단위 누수'
print(f'train {len(tr_i)} / valid {len(va_i)} / test {len(te_i)}   '
      f'양성률 {yy[tr_i].mean():.3f} / {yy[va_i].mean():.3f} / {yy[te_i].mean():.3f}')

json.dump({'train': tr_i.tolist(), 'valid': va_i.tolist(), 'test': te_i.tolist(),
           'removed_clinical_overlap': int((~mask).sum()), 'seed': SEED},
          open(GOUT + 'herg_karim_pretrain_splits.json', 'w', encoding='utf-8'))

mk = lambda ii, bs, sh: DataLoader([gs[i] for i in ii], batch_size=bs, shuffle=sh)
L_tr, L_va, L_te = mk(tr_i, BS_TR, True), mk(va_i, BS_EV, False), mk(te_i, BS_EV, False)

임상 중첩 209개 제거 → 10828개
train 8662 / valid 1083 / test 1083   양성률 0.493 / 0.493 / 0.493


In [7]:
# ── 셀 4 │ 모델 (SI Figure S1 / Note S1) ──
class GH(nn.Module):
    """단일 attention head (Veličković 식 1-3)."""
    def __init__(s, ic, oc, d=DROP):
        super().__init__()
        s.W = nn.Parameter(torch.empty(ic, oc)); s.a = nn.Parameter(torch.empty(2*oc, 1))
        nn.init.xavier_uniform_(s.W); nn.init.xavier_uniform_(s.a)
        s.lr = nn.LeakyReLU(0.2); s.d = d
    def forward(s, x, ei):
        ei, _ = add_self_loops(ei, num_nodes=x.size(0))
        h = x @ s.W; row, col = ei
        e = s.lr((torch.cat([h[row], h[col]], -1) @ s.a).squeeze(-1))
        e = F.dropout(e, s.d, training=s.training)
        al = softmax(e, row)
        o = torch.zeros_like(h); o.index_add_(0, row, h[col] * al.unsqueeze(-1))
        return F.elu(o)

class GL(nn.Module):
    """multi-head layer. h>1 concat(식 4), h==1 average(식 5)."""
    def __init__(s, ic, oc, h, d=DROP):
        super().__init__()
        s.h, s.d = h, d
        s.att = nn.ModuleList([GH(ic, oc, d) for _ in range(h)])
    def forward(s, x, ei):
        z = (torch.mean(torch.stack([a(x, ei) for a in s.att], 0), 0) if s.h == 1
             else torch.cat([a(x, ei) for a in s.att], -1))
        return F.dropout(z, s.d, training=s.training)

class AttentionPool(nn.Module):
    """그래프 단위 softmax gate 가중합."""
    def __init__(s, ch, d=DROP):
        super().__init__()
        s.gate = nn.Sequential(nn.Linear(ch, ch), nn.ReLU(), nn.Dropout(d), nn.Linear(ch, 1))
    def forward(s, x, batch):
        al = softmax(s.gate(x).squeeze(-1), batch)
        out = torch.zeros(int(batch.max()) + 1, x.size(1), device=x.device, dtype=x.dtype)
        out.index_add_(0, batch, x * al.unsqueeze(-1))
        return out

class GAT_SI(nn.Module):
    """3층 GAT (8, 8, 1 heads) + attention pooling + 2층 FC classifier."""
    def __init__(s, ic, hc=HID, h=HEADS, d=DROP):
        super().__init__()
        s.l1 = GL(ic, hc, h, d)
        s.l2 = GL(hc*h, hc, h, d)
        s.l3 = GL(hc*h, hc, 1, d)          # 32차원 노드 임베딩
        s.pool = AttentionPool(hc, d)
        s.clf = nn.Sequential(nn.Linear(hc, 64), nn.ReLU(), nn.Dropout(d), nn.Linear(64, 1))
    def backbone_parameters(s):
        return (list(s.l1.parameters()) + list(s.l2.parameters())
                + list(s.l3.parameters()) + list(s.pool.parameters()))
    def forward(s, x, ei, b):
        x = F.elu(s.l1(x, ei)); x = F.elu(s.l2(x, ei)); x = s.l3(x, ei)
        return s.clf(s.pool(x, b)).squeeze(-1)

_m = GAT_SI(ATOM_DIM)
print(f'파라미터 {sum(p.numel() for p in _m.parameters()):,}')
del _m

파라미터 88,066


In [13]:
# ── 셀 5 │ loss 3종 + 평가 지표 + 학습 루프 (완성본) ──
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             accuracy_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix)

# ---- loss ----------------------------------------------------------------
class FocalLoss(nn.Module):
    def __init__(s, alpha=0.25, gamma=2.0):
        super().__init__(); s.alpha, s.gamma = alpha, gamma
    def forward(s, logit, y):
        bce = F.binary_cross_entropy_with_logits(logit, y, reduction='none')
        p = torch.sigmoid(logit)
        pt = p*y + (1-p)*(1-y)
        at = s.alpha*y + (1-s.alpha)*(1-y)
        return (at * (1-pt)**s.gamma * bce).mean()

def make_loss(name, y_train):
    if name == 'bce':
        return nn.BCEWithLogitsLoss()
    if name == 'wbce':
        npos, nneg = max(int((y_train == 1).sum()), 1), max(int((y_train == 0).sum()), 1)
        print(f'   pos_weight = {nneg}/{npos} = {nneg/npos:.3f}')
        return nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor([nneg/npos], dtype=torch.float, device=DEV))
    if name == 'focal':
        return FocalLoss(0.25, 2.0)
    raise ValueError(name)

# ---- 지표 ----------------------------------------------------------------
def _at(ts, ps, th):
    pred = (ps >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(ts, pred, labels=[0, 1]).ravel()
    return {'threshold': float(th),
            'accuracy':  float(accuracy_score(ts, pred)),
            'bal_acc':   float(balanced_accuracy_score(ts, pred)),
            'f1':        float(f1_score(ts, pred, zero_division=0)),
            'precision': float(precision_score(ts, pred, zero_division=0)),
            'recall':    float(recall_score(ts, pred, zero_division=0)),
            'specificity': float(tn/(tn+fp)) if (tn+fp) else 0.0,
            'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)}

def best_threshold(ts, ps):
    grid = np.arange(0.05, 0.96, 0.02)
    return float(max(grid, key=lambda t: f1_score(ts, ps >= t, zero_division=0)))

@torch.no_grad()
def evaluate(model, ld, fixed_thr=None, full=True):
    """full=False 면 AUROC/AUCPR만 (학습 중 빠른 검증용)."""
    model.eval(); ps, ts = [], []
    for b in ld:
        b = b.to(DEV)
        ps += torch.sigmoid(model(b.x, b.edge_index, b.batch)).cpu().tolist()
        ts += b.y.view(-1).cpu().tolist()
    ps, ts = np.array(ps), np.array(ts, dtype=int)

    try: auroc = float(roc_auc_score(ts, ps))
    except ValueError: auroc = 0.5
    out = {'auroc': auroc, 'aucpr': float(average_precision_score(ts, ps)),
           'n': int(len(ts)), 'n_pos': int(ts.sum())}
    if full:
        out['at_050']   = _at(ts, ps, 0.5)
        out['at_f1opt'] = _at(ts, ps, best_threshold(ts, ps))
        if fixed_thr is not None:
            out['at_valthr'] = _at(ts, ps, fixed_thr)
    return out

# ---- 학습 ----------------------------------------------------------------
def fit(loss_name, verbose_every=5):
    seed_all(SEED)
    model = GAT_SI(ATOM_DIM).to(DEV)
    crit = make_loss(loss_name, yy[tr_i])
    opt = torch.optim.Adam(
        [{'params': model.backbone_parameters(), 'lr': LR_BB},
         {'params': model.clf.parameters(),      'lr': LR_CLF}], weight_decay=WD)

    best, best_state, wait, hist = 0.0, None, 0, []
    for ep in range(1, EPOCHS + 1):
        model.train(); tot = 0.0
        for b in L_tr:
            b = b.to(DEV); opt.zero_grad()
            loss = crit(model(b.x, b.edge_index, b.batch), b.y.view(-1).float())
            loss.backward(); opt.step(); tot += loss.item() * b.num_graphs

        v = evaluate(model, L_va, full=False)      # 에폭마다는 빠른 버전
        hist.append({'epoch': ep, 'loss': tot/len(tr_i),
                     'val_auroc': v['auroc'], 'val_aucpr': v['aucpr']})

        if v['auroc'] > best:
            best, wait = v['auroc'], 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1

        if ep % verbose_every == 0 or wait == 0:
            print(f'  ep{ep:>3} loss {tot/len(tr_i):.4f}  '
                  f'val AUROC {v["auroc"]:.4f}  AUCPR {v["aucpr"]:.4f}'
                  + ('  *' if wait == 0 else ''))

        if wait >= PATIENCE:
            print(f'  early stop @ {ep} (best val AUROC {best:.4f})'); break

    model.load_state_dict({k: t.to(DEV) for k, t in best_state.items()})
    return model, best, hist

In [14]:
# ── 셀 6 │ 교체본 ──
results = {}
for name in ['bce', 'wbce', 'focal']:
    print(f'\n{"="*64}\n[{name}]')
    t0 = datetime.datetime.now()
    model, best_val, hist = fit(name)

    va = evaluate(model, L_va)
    vthr = va['at_f1opt']['threshold']          # 임계값은 검증셋에서 고정
    te = evaluate(model, L_te, fixed_thr=vthr)
    mins = (datetime.datetime.now() - t0).total_seconds() / 60

    a5, af, av = te['at_050'], te['at_f1opt'], te['at_valthr']
    print(f'  test AUROC {te["auroc"]:.4f}  AUCPR {te["aucpr"]:.4f}')
    print(f'    @0.50     acc {a5["accuracy"]:.4f}  f1 {a5["f1"]:.4f}  '
          f'prec {a5["precision"]:.4f}  rec {a5["recall"]:.4f}')
    print(f'    @val{vthr:.2f}   acc {av["accuracy"]:.4f}  f1 {av["f1"]:.4f}  '
          f'prec {av["precision"]:.4f}  rec {av["recall"]:.4f}')
    print(f'    @f1opt{af["threshold"]:.2f} acc {af["accuracy"]:.4f}  f1 {af["f1"]:.4f}  '
          f'prec {af["precision"]:.4f}  rec {af["recall"]:.4f}   ({mins:.1f}분)')

    torch.save(model.state_dict(), CKPT + f'pretrain_herg_karim_{name}.pt')
    results[name] = {'best_val_auroc': best_val, 'valid': va, 'test': te,
                     'val_threshold': vthr, 'epochs_run': len(hist),
                     'minutes': round(mins, 1), 'history': hist}

json.dump({'config': {'seed': SEED, 'hidden': HID, 'heads': HEADS, 'dropout': DROP,
                      'arch': '3-layer GAT (8,8,1 heads) + attention pooling',
                      'atom_dim': ATOM_DIM, 'bond_dim': BOND_DIM,
                      'epochs': EPOCHS, 'patience': PATIENCE,
                      'lr_backbone': LR_BB, 'lr_clf': LR_CLF, 'weight_decay': WD,
                      'batch': [BS_TR, BS_EV],
                      'n_train': len(tr_i), 'n_valid': len(va_i), 'n_test': len(te_i),
                      'removed_clinical_overlap': int((~mask).sum()),
                      'threshold_policy': 'F1-optimal on validation, applied to test'},
           'results': results},
          open(CKPT + 'pretrain_herg_karim_results.json', 'w', encoding='utf-8'),
          indent=2, ensure_ascii=False)

# ── 논문 표 형식 요약 ──
def row(r, which):
    a = r['test'][which]
    return {'Accuracy': a['accuracy'], 'AUCROC': r['test']['auroc'],
            'AUCPR': r['test']['aucpr'], 'F1': a['f1'],
            'Precision': a['precision'], 'Recall': a['recall'],
            'thr': a['threshold']}

for which, title in [('at_050', '@ 0.50'), ('at_valthr', '@ validation-tuned')]:
    print(f'\n{"="*64}\n[test, {title}]')
    print(pd.DataFrame({k: row(v, which) for k, v in results.items()}).T.round(4).to_string())


[bce]
  ep  1 loss 0.6748  val AUROC 0.7284  AUCPR 0.6980  *
  ep  2 loss 0.6367  val AUROC 0.7391  AUCPR 0.7192  *
  ep  4 loss 0.6226  val AUROC 0.7431  AUCPR 0.7220  *
  ep  5 loss 0.6179  val AUROC 0.7469  AUCPR 0.7299  *
  ep  6 loss 0.6175  val AUROC 0.7485  AUCPR 0.7280  *
  ep  7 loss 0.6175  val AUROC 0.7494  AUCPR 0.7338  *
  ep  9 loss 0.6137  val AUROC 0.7524  AUCPR 0.7405  *
  ep 10 loss 0.6130  val AUROC 0.7528  AUCPR 0.7414  *
  ep 11 loss 0.6111  val AUROC 0.7551  AUCPR 0.7359  *
  ep 14 loss 0.6088  val AUROC 0.7553  AUCPR 0.7437  *
  ep 15 loss 0.6094  val AUROC 0.7569  AUCPR 0.7463  *
  ep 17 loss 0.6085  val AUROC 0.7589  AUCPR 0.7502  *
  ep 20 loss 0.6012  val AUROC 0.7561  AUCPR 0.7484
  ep 22 loss 0.6015  val AUROC 0.7639  AUCPR 0.7566  *
  ep 25 loss 0.5985  val AUROC 0.7651  AUCPR 0.7563  *
  ep 26 loss 0.5997  val AUROC 0.7662  AUCPR 0.7582  *
  ep 30 loss 0.5954  val AUROC 0.7670  AUCPR 0.7600  *
  ep 33 loss 0.5923  val AUROC 0.7702  AUCPR 0.7643  *
  ep 3

In [15]:
# ── 셀 7 │ 최종 요약 표 ──
ORDER = [('bce', 'BCE'), ('focal', 'Focal Loss'), ('wbce', 'Weighted BCE')]
COLS  = ['Accuracy', 'AUCROC', 'AUCPR', 'F1 score', 'Precision', 'Recall']

def summary_table(results, which='at_valthr', order=ORDER, split='test'):
    """which: 'at_valthr'(검증셋에서 고른 임계값, 권장) | 'at_050' | 'at_f1opt'"""
    rows = {}
    for key, label in order:
        if key not in results:
            continue
        r = results[key][split]
        a = r.get(which) or r['at_050']
        rows[label] = {'Accuracy':  a['accuracy'],
                       'AUCROC':    r['auroc'],
                       'AUCPR':     r['aucpr'],
                       'F1 score':  a['f1'],
                       'Precision': a['precision'],
                       'Recall':    a['recall']}
    return pd.DataFrame(rows).T[COLS].round(4)

tbl = summary_table(results, which='at_valthr')
print('[hERG_Karim pre-training — test set, 임계값은 검증셋에서 결정]')
print(tbl.to_string())

# 보조 표: 임계값 정책별 비교 (SI 용)
for which, title in [('at_050', '@ 0.50 고정'), ('at_f1opt', '@ test F1-optimal')]:
    print(f'\n[참고] {title}')
    print(summary_table(results, which=which).to_string())

# 저장 ─ 엑셀/워드에 그대로 붙여넣을 수 있게
tbl.to_csv(CKPT + 'table_pretrain_herg_karim.csv', encoding='utf-8-sig')
with open(CKPT + 'table_pretrain_herg_karim.md', 'w', encoding='utf-8') as f:
    f.write('# hERG_Karim pre-training (test set)\n\n')
    f.write(f'n_test = {results["bce"]["test"]["n"]}, '
            f'positives = {results["bce"]["test"]["n_pos"]}  \n')
    f.write('Threshold selected on the validation set (F1-optimal) '
            'and applied unchanged to the test set.\n\n')
    f.write(tbl.to_markdown())
    f.write('\n')
print('\n저장:', CKPT + 'table_pretrain_herg_karim.csv')

# 탭 구분 ─ 그대로 복사해서 엑셀/워드 표에 붙여넣기
print('\n' + '='*64 + '\n[탭 구분 복사용]')
print('\t' + '\t'.join(COLS))
for lab, r in tbl.iterrows():
    print(lab + '\t' + '\t'.join(f'{r[c]:.4f}' for c in COLS))

[hERG_Karim pre-training — test set, 임계값은 검증셋에서 결정]
              Accuracy  AUCROC   AUCPR  F1 score  Precision  Recall
BCE             0.6436  0.7807  0.7706    0.7080     0.5939  0.8764
Focal Loss      0.6260  0.7237  0.7001    0.6957     0.5809  0.8670
Weighted BCE    0.6436  0.7817  0.7783    0.7115     0.5920  0.8914

[참고] @ 0.50 고정
              Accuracy  AUCROC   AUCPR  F1 score  Precision  Recall
BCE             0.7091  0.7807  0.7706    0.7020     0.7094  0.6948
Focal Loss      0.5069  0.7237  0.7001    0.0000     0.0000  0.0000
Weighted BCE    0.7045  0.7817  0.7783    0.6992     0.7019  0.6966

[참고] @ test F1-optimal
              Accuracy  AUCROC   AUCPR  F1 score  Precision  Recall
BCE             0.7008  0.7807  0.7706    0.7221     0.6661  0.7884
Focal Loss      0.6260  0.7237  0.7001    0.6957     0.5809  0.8670
Weighted BCE    0.6999  0.7817  0.7783    0.7271     0.6591  0.8109


ImportError: Missing optional dependency 'tabulate'.  Use pip or conda to install tabulate.